In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
print(f"Project root directory: {PROJECT_ROOT}")
from src.data.prepare import load_partition
from src.preprocessing.pipelines import task_data, make_model_pipeline

Project root directory: C:\Users\Mostafa\Desktop\Fall26\Fall26-MACHINE-LEARNING\project\hotel-pms-predictive-suite


In [3]:
train_df = load_partition('train')
val_df = load_partition('validation')

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

Train shape: (31921, 31)
Val shape: (10650, 31)


In [4]:
from sklearn.linear_model import Ridge

X_train, y_train = task_data(train_df, 'regression')
X_val, y_val = task_data(val_df, 'regression')

# Fit the end-to-end pipeline (cleaning + encoding + scaling + Ridge)
baseline = make_model_pipeline('regression', Ridge())
baseline.fit(X_train, y_train)

# Test prediction on validation
preds = baseline.predict(X_val)
print("First 5 predictions:", preds[:5])

First 5 predictions: [110.06039621 111.84316297 108.06883492  68.34730081  94.50535081]


In [6]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
import pandas as pd

# 1. Dummy baseline (predicts training mean)
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_val)

# 2. Evaluate
def get_metrics(y_true, y_pred, name):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame([
    get_metrics(y_val, dummy_preds, "Dummy (Mean)"),
    get_metrics(y_val, preds, "Ridge Regression (Baseline)")
])
print(results)

                         Model        MAE       RMSE        R2
0                 Dummy (Mean)  31.976380  41.940842 -0.000124
1  Ridge Regression (Baseline)  16.609467  23.627849  0.682585
